# LoRA 微调 + torch.profiler（Colab / PEFT / TRL）

基于 Hugging Face **Transformers** 生态的 LoRA SFT 示例：

- **PEFT** `LoraConfig` — 配置 LoRA adapter
- **TRL** `SFTTrainer` — 监督微调
- **torch.profiler** — 采集 CPU/CUDA trace，导出 Chrome Trace JSON
- **Colab 下载** — 打包 zip 后一键下载 profile 文件

> 运行时请选择 **GPU**（T4 / L4 / A100）。
>
> Profile 文件可用 [Perfetto UI](https://ui.perfetto.dev/) 或 Chrome `chrome://tracing` 打开。

In [ ]:
# @title 1. 安装依赖（Transformers + PEFT + TRL）
!pip install -q "transformers>=4.46.0" "datasets>=3.0.0" "accelerate>=1.0.0" \
    "peft>=0.13.0" "trl>=0.12.0" "huggingface_hub>=0.26.0"

In [ ]:
# @title 2. 导入 & 环境检查
import json
import shutil
from pathlib import Path

import torch
from datasets import load_dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainerCallback
from trl import SFTConfig, SFTTrainer

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import transformers

assert torch.cuda.is_available(), "请在 Colab 中选择 GPU 运行时。"
print(f"transformers: {transformers.__version__}")
print(f"torch       : {torch.__version__}")
print(f"GPU         : {torch.cuda.get_device_name(0)}")
print(f"Colab       : {IN_COLAB}")

In [ ]:
# @title 3. 实验 & Profiler 配置
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
OUTPUT_DIR = "./lora-output"
PROFILE_DIR = Path("./torch_profiler_traces")
PROFILE_DIR.mkdir(parents=True, exist_ok=True)

MAX_SAMPLES = 128
MAX_LENGTH = 512
TRAIN_MAX_STEPS = 10          # 仅跑少量 step，便于 Colab 快速 profile

LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    max_steps=TRAIN_MAX_STEPS,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="no",
    max_length=MAX_LENGTH,
    packing=False,
    gradient_checkpointing=True,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    report_to="none",
)

# torch.profiler schedule: wait=1, warmup=1, active=3 → 导出 3 个 active step 的 trace
PROFILER_SCHEDULE = torch.profiler.schedule(wait=1, warmup=1, active=3, repeat=1)
EXPORTED_TRACES: list[Path] = []


def trace_handler(prof: torch.profiler.profile) -> None:
    """每个 active step 结束时导出 Chrome Trace JSON。"""
    trace_path = PROFILE_DIR / f"lora_trace_step_{prof.step_num:03d}.json"
    prof.export_chrome_trace(str(trace_path))
    EXPORTED_TRACES.append(trace_path)
    print(f"[profiler] exported {trace_path}")


class TorchProfilerCallback(TrainerCallback):
    """在 Trainer 每个 step 结束时驱动 torch.profiler.step()。"""

    def __init__(self, profiler: torch.profiler.profile):
        self.profiler = profiler

    def on_step_end(self, args, state, control, **kwargs):
        self.profiler.step()

print(f"Profile output dir: {PROFILE_DIR.resolve()}")

In [ ]:
# @title 4. 加载数据集
train_dataset = load_dataset("trl-lib/Capybara", split=f"train[:{MAX_SAMPLES}]")
print(train_dataset)
print(train_dataset[0]["messages"][:2])

In [ ]:
# @title 5. 加载模型 & 构建 SFTTrainer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
if training_args.gradient_checkpointing:
    model.enable_input_require_grads()

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
)

trainer.model.print_trainable_parameters()

In [ ]:
# @title 6. 带 torch.profiler 的 LoRA 微调
with torch.profiler.profile(
    activities=[
        torch.profiler.ProfilerActivity.CPU,
        torch.profiler.ProfilerActivity.CUDA,
    ],
    schedule=PROFILER_SCHEDULE,
    on_trace_ready=trace_handler,
    record_shapes=True,
    profile_memory=True,
    with_stack=True,
) as prof:
    trainer.add_callback(TorchProfilerCallback(prof))
    train_result = trainer.train()

print(train_result)

# 保存汇总表
summary_path = PROFILE_DIR / "key_averages.txt"
summary_table = prof.key_averages(group_by_stack_n=5).table(
    sort_by="self_cuda_time_total",
    row_limit=30,
)
summary_path.write_text(summary_table, encoding="utf-8")
print(f"\nSaved summary: {summary_path}")
print(summary_table)

# 保存 memory timeline（PyTorch 2.1+）
timeline_path = PROFILE_DIR / "memory_timeline.html"
try:
    prof.export_memory_timeline(str(timeline_path), device="cuda:0")
    print(f"Memory timeline saved: {timeline_path}")
except Exception as exc:
    print(f"[skip] memory timeline: {exc}")

# 保存 metadata
meta = {
    "model_id": MODEL_ID,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "max_steps": TRAIN_MAX_STEPS,
    "max_length": MAX_LENGTH,
    "batch_size": training_args.per_device_train_batch_size,
    "grad_accum": training_args.gradient_accumulation_steps,
    "exported_traces": [p.name for p in EXPORTED_TRACES],
    "gpu": torch.cuda.get_device_name(0),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
}
(PROFILE_DIR / "run_metadata.json").write_text(
    json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8"
)

In [ ]:
# @title 7. 打包 & 从 Colab 下载 profile 文件
zip_path = Path("lora_torch_profiler_traces.zip")
if zip_path.exists():
    zip_path.unlink()

shutil.make_archive(zip_path.stem, "zip", root_dir=PROFILE_DIR.parent, base_dir=PROFILE_DIR.name)
print(f"Created archive: {zip_path.resolve()} ({zip_path.stat().st_size / 1e6:.1f} MB)")
print("\nArchive contents:")
for p in sorted(PROFILE_DIR.iterdir()):
    print(f"  - {p.name} ({p.stat().st_size / 1e3:.1f} KB)")

if IN_COLAB:
    files.download(str(zip_path))
    print("\n✅ Download started. Open *.json traces in https://ui.perfetto.dev/")
else:
    print("\nNot in Colab — zip saved locally.")

In [ ]:
# @title 8. （可选）保存 adapter & 快速推理
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

prompt = "用一句话解释 LoRA 是什么？"
messages = [{"role": "user", "content": prompt}]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to(model.device)

model.eval()
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=64, do_sample=False)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
# @title 9. （可选）单独下载某个 trace 文件
if IN_COLAB and EXPORTED_TRACES:
    sample_trace = EXPORTED_TRACES[0]
    print(f"Downloading {sample_trace.name} ...")
    files.download(str(sample_trace))
else:
    print("Run cell 6 first, or open traces from the zip archive.")

## Profile 文件说明

| 文件 | 用途 |
| --- | --- |
| `lora_trace_step_XXX.json` | Chrome Trace，在 Perfetto / chrome://tracing 中查看算子耗时 |
| `key_averages.txt` | 按 CUDA 时间排序的算子汇总表 |
| `memory_timeline.html` | 显存时间线（若 PyTorch 版本支持） |
| `run_metadata.json` | 实验超参与环境信息 |
| `lora_torch_profiler_traces.zip` | 以上全部打包，Colab 一键下载 |

**查看 trace：** 打开 https://ui.perfetto.dev/ → Open trace file → 选择 `lora_trace_step_*.json`